# 26 - Diagnose Triton Latency Tail

Diagnóstico da cauda de latência observada no notebook 25.

Hipótese atual:
- a cauda de ~22s vem de compilação/warmup por shape, não de inferência steady-state.

Objetivos:
1. separar cold/warm por pergunta e por comprimento de prompt;
2. medir primeira execução vs repetições seguintes;
3. agrupar por `prompt_tokens` e `generated_tokens`;
4. registrar se o problema some após pré-warm global.


In [ ]:
# Dependências principais.
import os, sys, subprocess, importlib.util, importlib.metadata

required = {
    'torch': 'torch',
    'triton': 'triton',
    'transformers': 'transformers',
    'accelerate': 'accelerate',
    'peft': 'peft',
    'safetensors': 'safetensors',
    'sentencepiece': 'sentencepiece',
    'psutil': 'psutil',
    'packaging': 'packaging',
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Instalando dependências ausentes:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

from packaging.version import Version
def installed_version(package_name):
    try:
        return Version(importlib.metadata.version(package_name))
    except importlib.metadata.PackageNotFoundError:
        return None

torchao_version = installed_version('torchao')
if torchao_version is not None:
    print(f'Removendo torchao para evitar warnings/side effects no benchmark: {torchao_version}')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)

import torch
import psutil

def human_bytes(n):
    n = float(n)
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024.0 or unit == 'TB':
            return f'{n:.2f} {unit}'
        n /= 1024.0

print('Dependências prontas.')


In [ ]:
# Bootstrap Colab/local e paths oficiais.
from pathlib import Path
import json, time, gc, shutil, platform

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)

CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
EXPORTS_DIR = PROJECT_ROOT / 'exports'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'packed_recovery'
BUILD_DIR = PROJECT_ROOT / 'build'
RUNTIME_DIR = PROJECT_ROOT / 'runtime' / 'packed_recovered_l78'
DIST_DIR = BUILD_DIR / 'distributable_packed_recovered_l78'
DATA_PATH = PROJECT_ROOT / 'data' / 'gemma4_100_questions.json'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BUILD_DIR.mkdir(parents=True, exist_ok=True)
DIST_DIR.mkdir(parents=True, exist_ok=True)
if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RUNTIME_DIR =', RUNTIME_DIR)
print('DIST_DIR =', DIST_DIR)

In [ ]:
# Garante que a pasta distribuível contenha o runtime Triton atualizado.
# Diferente do 21/22, este notebook se autocorrige: se o Drive tiver runtime antigo,
# baixa os arquivos versionados do GitHub antes de montar o pacote.
import urllib.request

STUDENT_DIR = CHECKPOINTS_DIR / 'student_pruned'
ADAPTER_DIR = CHECKPOINTS_DIR / 'student_packed_ternary_recovered_l78_t4_resume'
PACKED_SOURCE_DIR = next((p for p in [EXPORTS_DIR / 'ternary_mlp_l78', DEPLOY_DIR] if (p / 'manifest.json').exists() and (p / 'mlp_ternary_2bit.bin').exists()), None)
RUNTIME_RAW_BASE = 'https://raw.githubusercontent.com/cesarfavero/RPT/main/gemma4_theory/runtime/packed_recovered_l78'
RUNTIME_FILES = [
    'README.md',
    'requirements.txt',
    'loader.py',
    'run_inference.py',
    'native_loader.py',
    'run_native_inference.py',
    'triton_loader.py',
    'run_triton_inference.py',
    'package_manifest.template.json',
]

def copytree_clean(src, dst):
    src = Path(src)
    dst = Path(dst)
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    shutil.copytree(src, dst)

def copy_file_clean(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    shutil.copy2(src, dst)

def download_runtime_file(filename):
    url = f'{RUNTIME_RAW_BASE}/{filename}'
    target = RUNTIME_DIR / filename
    target.parent.mkdir(parents=True, exist_ok=True)
    print('baixando runtime ausente:', filename)
    with urllib.request.urlopen(url, timeout=30) as response:
        content = response.read()
    target.write_bytes(content)

def ensure_runtime_templates():
    RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
    missing = [filename for filename in RUNTIME_FILES if not (RUNTIME_DIR / filename).exists()]
    for filename in missing:
        download_runtime_file(filename)
    still_missing = [str(RUNTIME_DIR / filename) for filename in RUNTIME_FILES if not (RUNTIME_DIR / filename).exists()]
    if still_missing:
        raise FileNotFoundError(
            'Runtime continua incompleto após auto-download:\n' + '\n'.join(still_missing)
        )
    return missing

def ensure_distributable():
    if PACKED_SOURCE_DIR is None:
        raise FileNotFoundError('Packed source not found in exports/ternary_mlp_l78 or deploy')
    runtime_downloaded = ensure_runtime_templates()
    required_sources = [STUDENT_DIR, ADAPTER_DIR, RUNTIME_DIR]
    missing = [str(p) for p in required_sources if not Path(p).exists()]
    if missing:
        raise FileNotFoundError('Missing sources:\n' + '\n'.join(missing))

    for src, dst in [(STUDENT_DIR, DIST_DIR / 'student_pruned'), (ADAPTER_DIR, DIST_DIR / 'adapter_recovered_l78'), (RUNTIME_DIR, DIST_DIR / 'runtime')]:
        copytree_clean(src, dst)
    (DIST_DIR / 'packed').mkdir(parents=True, exist_ok=True)
    copy_file_clean(PACKED_SOURCE_DIR / 'manifest.json', DIST_DIR / 'packed' / 'manifest.json')
    copy_file_clean(PACKED_SOURCE_DIR / 'mlp_ternary_2bit.bin', DIST_DIR / 'packed' / 'mlp_ternary_2bit.bin')
    return runtime_downloaded

runtime_downloaded = ensure_distributable()
runtime_dist = DIST_DIR / 'runtime'
required_runtime_files = [
    runtime_dist / 'triton_loader.py',
    runtime_dist / 'run_triton_inference.py',
    runtime_dist / 'native_loader.py',
]
missing_runtime = [str(p) for p in required_runtime_files if not p.exists()]
if missing_runtime:
    raise FileNotFoundError(
        'Runtime distribuível continua desatualizado. Ausentes:\n' + '\n'.join(missing_runtime)
    )

# Garante prioridade do runtime distribuível e evita módulos antigos cacheados em sessão Colab.
runtime_dist_str = str(runtime_dist)
if runtime_dist_str in sys.path:
    sys.path.remove(runtime_dist_str)
sys.path.insert(0, runtime_dist_str)
for module_name in ['triton_loader', 'native_loader', 'loader']:
    if module_name in sys.modules:
        del sys.modules[module_name]

print('distributable ready:', DIST_DIR)
print('runtime_dist =', runtime_dist)
print('runtime_downloaded =', runtime_downloaded)
print('runtime files:', sorted(p.name for p in runtime_dist.glob('*.py')))


In [ ]:
# Configuração do benchmark.
BENCH_OUT_DIR = RESULTS_DIR / 'runtime_benchmarks'
BENCH_OUT_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_QUESTIONS = [
    'Qual a capital da França?',
    'Quanto é 12 x 8?',
    'Quem desenvolveu a teoria da relatividade?',
    'Em Python, qual palavra-chave define uma função?',
    'Se hoje é segunda-feira, que dia será depois de 3 dias?',
    'Qual esporte usa raquete, rede e peteca?',
    'Qual gás as plantas absorvem principalmente para fazer fotossíntese?',
    'Qual comando Linux lista arquivos de um diretório?',
]

WARMUP_ROUNDS = 1
MEASURE_ROUNDS = 3
MAX_NEW_TOKENS = 12
PROMPT_TEMPLATE = 'Responda de forma curta e direta.\nPergunta: {question}\nResposta:'

# DIST_DIR é reconstruído automaticamente na célula anterior.
BUILD_CANDIDATES = [
    DIST_DIR,
    BUILD_DIR / 'packed_recovered_l78',
]
MODEL_DIRS = []
for p in BUILD_CANDIDATES:
    if p.exists() and (p / 'runtime').exists() and (p / 'packed').exists():
        MODEL_DIRS.append(p)
if not MODEL_DIRS:
    raise FileNotFoundError('Nenhum build pronto encontrado após auto-rebuild. Esperado: runtime/ e packed/ em build/distributable_packed_recovered_l78 ou build/packed_recovered_l78')
print('MODEL_DIRS =', [str(p) for p in MODEL_DIRS])
print('runtime_downloaded =', runtime_downloaded)


In [ ]:
import statistics


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def snapshot_memory(label):
    rss = psutil.Process().memory_info().rss
    item = {
        'label': label,
        'rss_bytes': rss,
        'rss_human': human_bytes(rss),
    }
    if torch.cuda.is_available():
        item.update({
            'cuda_device': torch.cuda.get_device_name(0),
            'cuda_allocated_bytes': torch.cuda.memory_allocated(),
            'cuda_reserved_bytes': torch.cuda.memory_reserved(),
            'cuda_max_allocated_bytes': torch.cuda.max_memory_allocated(),
            'cuda_allocated_human': human_bytes(torch.cuda.memory_allocated()),
            'cuda_reserved_human': human_bytes(torch.cuda.memory_reserved()),
            'cuda_max_allocated_human': human_bytes(torch.cuda.max_memory_allocated()),
        })
    return item


def percentile(values, q):
    if not values:
        return None
    if len(values) == 1:
        return values[0]
    vals = sorted(values)
    idx = (len(vals) - 1) * q
    lo = int(idx)
    hi = min(lo + 1, len(vals) - 1)
    frac = idx - lo
    return vals[lo] * (1 - frac) + vals[hi] * frac


def extract_answer(prompt, generated):
    text = generated
    if prompt in text:
        text = text.split(prompt, 1)[-1]
    return text.strip().split('\n')[0].strip()


In [ ]:
# Carregamento do runtime do build.
def load_runtime_model(model_dir):
    runtime_dir = Path(model_dir) / 'runtime'
    if not runtime_dir.exists():
        raise FileNotFoundError(f'Runtime ausente em {runtime_dir}')
    runtime_str = str(runtime_dir)
    if runtime_str in sys.path:
        sys.path.remove(runtime_str)
    sys.path.insert(0, runtime_str)
    for module_name in ['triton_loader', 'native_loader', 'loader']:
        if module_name in sys.modules:
            del sys.modules[module_name]

    import importlib

    candidate_calls = []

    try:
        triton_loader = importlib.import_module('triton_loader')
        fn = getattr(triton_loader, 'load_triton_packed_recovered_l78', None)
        if fn is not None:
            candidate_calls.append(('triton_loader.load_triton_packed_recovered_l78', fn))
    except Exception as exc:
        print('Aviso: falha ao importar triton_loader:', repr(exc))

    try:
        native_loader = importlib.import_module('native_loader')
        fn = getattr(native_loader, 'load_native_packed_recovered_l78', None)
        if fn is not None:
            candidate_calls.append(('native_loader.load_native_packed_recovered_l78', fn))
    except Exception as exc:
        print('Aviso: falha ao importar native_loader:', repr(exc))

    try:
        loader = importlib.import_module('loader')
        for name in ['load_packed_recovered_l78', 'load_model', 'load_runtime_model']:
            fn = getattr(loader, name, None)
            if fn is not None:
                candidate_calls.append((f'loader.{name}', fn))
    except Exception as exc:
        print('Aviso: falha ao importar loader:', repr(exc))

    if not candidate_calls:
        raise AttributeError('Nenhum loader compatível encontrado em triton_loader.py, native_loader.py ou loader.py')

    last_error = None
    for label, load_fn in candidate_calls:
        try:
            t0 = time.perf_counter()
            model, tokenizer, report = load_fn(str(model_dir))
            load_elapsed = time.perf_counter() - t0
            report = dict(report)
            report['selected_loader'] = label
            model.eval()
            print('selected_loader =', label)
            return model, tokenizer, report, load_elapsed
        except TypeError:
            try:
                t0 = time.perf_counter()
                model, tokenizer, report = load_fn(str(model_dir), token=os.environ.get('HF_TOKEN'))
                load_elapsed = time.perf_counter() - t0
                report = dict(report)
                report['selected_loader'] = label
                model.eval()
                print('selected_loader =', label)
                return model, tokenizer, report, load_elapsed
            except Exception as exc:
                last_error = exc
                print('Loader falhou:', label, repr(exc))
        except Exception as exc:
            last_error = exc
            print('Loader falhou:', label, repr(exc))

    raise RuntimeError(f'Nenhum loader conseguiu carregar o modelo. Último erro: {last_error!r}')


In [ ]:
def benchmark_model_dir(model_dir):
    cleanup_memory()
    before = snapshot_memory('before_load')
    model, tokenizer, runtime_report, load_elapsed = load_runtime_model(model_dir)
    after = snapshot_memory('after_load')

    warmup_rows = []
    for question in BENCHMARK_QUESTIONS[:WARMUP_ROUNDS]:
        prompt = PROMPT_TEMPLATE.format(question=question)
        inputs = tokenizer(prompt, return_tensors='pt').to(next(model.parameters()).device)
        with torch.no_grad():
            t0 = time.perf_counter()
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
            elapsed = time.perf_counter() - t0
        generated = tokenizer.decode(out[0], skip_special_tokens=True)
        answer = extract_answer(prompt, generated)
        generated_tokens = max(0, out.shape[-1] - inputs['input_ids'].shape[-1])
        warmup_rows.append({'question': question, 'answer': answer, 'elapsed_sec': elapsed, 'generated_tokens': generated_tokens})

    measure_rows = []
    for question in BENCHMARK_QUESTIONS:
        prompt = PROMPT_TEMPLATE.format(question=question)
        for rep in range(MEASURE_ROUNDS):
            inputs = tokenizer(prompt, return_tensors='pt').to(next(model.parameters()).device)
            with torch.no_grad():
                t0 = time.perf_counter()
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
                elapsed = time.perf_counter() - t0
            generated = tokenizer.decode(out[0], skip_special_tokens=True)
            answer = extract_answer(prompt, generated)
            generated_tokens = max(0, out.shape[-1] - inputs['input_ids'].shape[-1])
            measure_rows.append({
                'question': question,
                'rep': rep + 1,
                'answer': answer,
                'elapsed_sec': elapsed,
                'generated_tokens': generated_tokens,
                'tokens_per_sec': (generated_tokens / elapsed) if elapsed > 0 and generated_tokens > 0 else None,
            })

    latencies = [r['elapsed_sec'] for r in measure_rows]
    tps = [r['tokens_per_sec'] for r in measure_rows if r['tokens_per_sec'] is not None]
    total_tokens = sum(r['generated_tokens'] for r in measure_rows)
    total_elapsed = sum(r['elapsed_sec'] for r in measure_rows)

    summary = {
        'model_dir': str(model_dir),
        'load_elapsed_sec': load_elapsed,
        'warmup_rounds': WARMUP_ROUNDS,
        'measure_rounds': MEASURE_ROUNDS,
        'num_questions': len(BENCHMARK_QUESTIONS),
        'num_measured_generations': len(measure_rows),
        'total_generated_tokens': total_tokens,
        'total_elapsed_sec': total_elapsed,
        'avg_tokens_per_sec_weighted': (total_tokens / total_elapsed) if total_elapsed > 0 else None,
        'avg_tokens_per_sec_unweighted': statistics.mean(tps) if tps else None,
        'latency_p50_sec': percentile(latencies, 0.50),
        'latency_p90_sec': percentile(latencies, 0.90),
        'latency_p95_sec': percentile(latencies, 0.95),
        'latency_min_sec': min(latencies) if latencies else None,
        'latency_max_sec': max(latencies) if latencies else None,
    }
    result = {
        'model_dir': str(model_dir),
        'runtime_report': runtime_report,
        'memory_before_load': before,
        'memory_after_load': after,
        'warmup_rows': warmup_rows,
        'measure_rows': measure_rows,
        'summary': summary,
    }
    del model, tokenizer
    cleanup_memory()
    return result


In [ ]:
# Configuração do diagnóstico.
BENCH_OUT_DIR = RESULTS_DIR / 'runtime_benchmarks'
BENCH_OUT_DIR.mkdir(parents=True, exist_ok=True)

DIAG_QUESTIONS = [
    'Qual a capital da França?',
    'Quanto é 12 x 8?',
    'Quem desenvolveu a teoria da relatividade?',
    'Em Python, qual palavra-chave define uma função?',
    'Se hoje é segunda-feira, que dia será depois de 3 dias?',
    'Qual esporte usa raquete, rede e peteca?',
    'Qual gás as plantas absorvem principalmente para fazer fotossíntese?',
    'Qual comando Linux lista arquivos de um diretório?',
]

PREWARM_PROMPTS = [
    'Qual a capital da França?',
    'Quanto é 12 x 8?',
    'Qual comando Linux lista arquivos de um diretório?',
]

MEASURE_REPS = 4
MAX_NEW_TOKENS = 12
PROMPT_TEMPLATE = 'Responda de forma curta e direta.\nPergunta: {question}\nResposta:'
MODEL_DIR = DIST_DIR
if not MODEL_DIR.exists():
    raise FileNotFoundError(f'Build distribuível ausente após auto-rebuild: {MODEL_DIR}')
print('MODEL_DIR =', MODEL_DIR)


In [ ]:
def run_generation(model, tokenizer, question):
    prompt = PROMPT_TEMPLATE.format(question=question)
    inputs = tokenizer(prompt, return_tensors='pt').to(next(model.parameters()).device)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    generated = tokenizer.decode(out[0], skip_special_tokens=True)
    answer = extract_answer(prompt, generated)
    prompt_tokens = int(inputs['input_ids'].shape[-1])
    total_tokens = int(out.shape[-1])
    generated_tokens = max(0, total_tokens - prompt_tokens)
    return {
        'question': question,
        'prompt_tokens': prompt_tokens,
        'generated_tokens': generated_tokens,
        'answer': answer,
        'elapsed_sec': elapsed,
        'tokens_per_sec': (generated_tokens / elapsed) if elapsed > 0 and generated_tokens > 0 else None,
    }


def summarize_rows(rows):
    lat = [r['elapsed_sec'] for r in rows]
    tps = [r['tokens_per_sec'] for r in rows if r['tokens_per_sec'] is not None]
    return {
        'count': len(rows),
        'latency_p50_sec': percentile(lat, 0.5),
        'latency_p90_sec': percentile(lat, 0.9),
        'latency_p95_sec': percentile(lat, 0.95),
        'latency_min_sec': min(lat) if lat else None,
        'latency_max_sec': max(lat) if lat else None,
        'avg_tps_unweighted': statistics.mean(tps) if tps else None,
    }


In [ ]:
cleanup_memory()
before = snapshot_memory('before_load')
model, tokenizer, runtime_report, load_elapsed = load_runtime_model(MODEL_DIR)
after = snapshot_memory('after_load')

# Fase 1: medir cold por pergunta, sem prewarm extra.
cold_rows = []
for question in DIAG_QUESTIONS:
    for rep in range(MEASURE_REPS):
        row = run_generation(model, tokenizer, question)
        row['rep'] = rep + 1
        row['phase'] = 'cold_sequence'
        cold_rows.append(row)

# Fase 2: prewarm controlado e medir de novo.
for question in PREWARM_PROMPTS:
    _ = run_generation(model, tokenizer, question)

warm_rows = []
for question in DIAG_QUESTIONS:
    for rep in range(MEASURE_REPS):
        row = run_generation(model, tokenizer, question)
        row['rep'] = rep + 1
        row['phase'] = 'post_prewarm'
        warm_rows.append(row)

# Agregações.
def group_first_vs_rest(rows):
    first = [r for r in rows if r['rep'] == 1]
    rest = [r for r in rows if r['rep'] > 1]
    return {'rep1': summarize_rows(first), 'rep_gt1': summarize_rows(rest)}

by_prompt_tokens = {}
for row in cold_rows + warm_rows:
    key = f"prompt_tokens={row['prompt_tokens']}|phase={row['phase']}"
    by_prompt_tokens.setdefault(key, []).append(row)
by_prompt_tokens = {k: summarize_rows(v) for k, v in by_prompt_tokens.items()}

payload = {
    'notebook': '26_diagnose_triton_latency_tail',
    'purpose': 'Diagnose whether Triton latency tail comes from compile/warmup per shape.',
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'model_dir': str(MODEL_DIR),
    'load_elapsed_sec': load_elapsed,
    'runtime_report': runtime_report,
    'memory_before_load': before,
    'memory_after_load': after,
    'cold_rows': cold_rows,
    'warm_rows': warm_rows,
    'cold_summary': summarize_rows(cold_rows),
    'warm_summary': summarize_rows(warm_rows),
    'cold_first_vs_rest': group_first_vs_rest(cold_rows),
    'warm_first_vs_rest': group_first_vs_rest(warm_rows),
    'by_prompt_tokens': by_prompt_tokens,
}
out_path = BENCH_OUT_DIR / f"26_diagnose_triton_latency_tail_{time.strftime('%Y%m%d_%H%M%S')}.json"
out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved', out_path)
print(json.dumps({
    'cold_summary': payload['cold_summary'],
    'warm_summary': payload['warm_summary'],
    'cold_first_vs_rest': payload['cold_first_vs_rest'],
    'warm_first_vs_rest': payload['warm_first_vs_rest'],
    'by_prompt_tokens': payload['by_prompt_tokens'],
}, ensure_ascii=False, indent=2)[:12000])
